# Phase 1 — Step 1: Generate Server Logs

Creates 15,000 fake SQL query logs that mimic a real legacy server — including **two deliberately injected problems** that we will discover in the next notebook.

**Stack:** Polars · DuckDB  
**Output:** `data/apex.duckdb` → table `raw_logs`  |  `data/synthetic_sql_logs.csv`

In [1]:
import polars as pl
import duckdb
import random
import os
from datetime import datetime, timedelta

random.seed(42)
os.makedirs("../data", exist_ok=True)
print("polars :", pl.__version__)
print("duckdb :", duckdb.__version__)


polars : 1.39.0
duckdb : 1.5.0


In [2]:
N_LOGS      = 15_000
START_DATE  = datetime(2024, 1, 1)
TABLES      = ["orders", "customers", "products", "order_items", "order_reviews"]
QTYPES      = ["SELECT", "INSERT", "UPDATE", "DELETE"]
TIMEOUT_MS  = 2_000
DB_PATH     = "../data/apex.duckdb"
CSV_PATH    = "../data/synthetic_sql_logs.csv"
print("Config ready.")


Config ready.


In [3]:
records = []
for _ in range(N_LOGS):
    ts    = START_DATE + timedelta(
                days=random.randint(0, 29),
                hours=random.randint(0, 23),
                minutes=random.randint(0, 59),
                seconds=random.randint(0, 59))
    table = random.choice(TABLES)
    qtype = random.choice(QTYPES)
    base  = random.randint(50, 300)

    # ── Injected problem 1: orders table has missing index → very slow ──
    if table == "orders" and random.random() < 0.15:
        exec_ms = base * random.randint(5, 35)
    # ── Injected problem 2: evening traffic peak → 50-90% slower ──────
    elif 18 <= ts.hour <= 22:
        exec_ms = int(base * (1.5 + random.random() * 0.4))
    else:
        exec_ms = base + random.randint(-20, 50)

    exec_ms = max(10, exec_ms)
    cpu     = round(min(98, 40 + (exec_ms / 200) * 30 + random.uniform(-5, 15)), 2)

    records.append({
        "timestamp"        : ts.strftime("%Y-%m-%d %H:%M:%S"),
        "query_type"       : qtype,
        "target_table"     : table,
        "execution_time_ms": exec_ms,
        "server_cpu_load"  : cpu,
        "status"           : "TIMEOUT" if exec_ms > TIMEOUT_MS else "SUCCESS",
    })

df = pl.DataFrame(records)
print(f"Generated  {len(df):,} rows")
print(df.head(3))


Generated  15,000 rows
shape: (3, 6)
┌─────────────────────┬────────────┬───────────────┬───────────────────┬─────────────────┬─────────┐
│ timestamp           ┆ query_type ┆ target_table  ┆ execution_time_ms ┆ server_cpu_load ┆ status  │
│ ---                 ┆ ---        ┆ ---           ┆ ---               ┆ ---             ┆ ---     │
│ str                 ┆ str        ┆ str           ┆ i64               ┆ f64             ┆ str     │
╞═════════════════════╪════════════╪═══════════════╪═══════════════════╪═════════════════╪═════════╡
│ 2024-01-21 03:01:47 ┆ INSERT     ┆ products      ┆ 104               ┆ 65.33           ┆ SUCCESS │
│ 2024-01-22 23:57:34 ┆ DELETE     ┆ orders        ┆ 638               ┆ 98.0            ┆ SUCCESS │
│ 2024-01-20 00:35:12 ┆ DELETE     ┆ order_reviews ┆ 143               ┆ 68.24           ┆ SUCCESS │
└─────────────────────┴────────────┴───────────────┴───────────────────┴─────────────────┴─────────┘


In [4]:
# ── DuckDB (primary storage) ──────────────────────────────────────────────────
with duckdb.connect(DB_PATH) as conn:
    conn.register("df", df)
    conn.execute("CREATE OR REPLACE TABLE raw_logs AS SELECT * FROM df")

# ── CSV (Power BI fallback) ───────────────────────────────────────────────────
df.write_csv(CSV_PATH)

print(f"DuckDB  →  {DB_PATH}  (table: raw_logs,  {len(df):,} rows)")
print(f"CSV     →  {CSV_PATH}")


DuckDB  →  ../data/apex.duckdb  (table: raw_logs,  15,000 rows)
CSV     →  ../data/synthetic_sql_logs.csv


In [5]:
conn = duckdb.connect(DB_PATH, read_only=True)   # read-only — safe

total    = conn.execute("SELECT COUNT(*) FROM raw_logs").fetchone()[0]
timeouts = conn.execute("SELECT COUNT(*) FROM raw_logs WHERE status='TIMEOUT'").fetchone()[0]

print(f"Total rows  : {total:,}")
print(f"Timeouts    : {timeouts:,}  ({timeouts/total:.2%})")
print()

print("── Anomaly 1: orders table vs others ──")
print(conn.execute('''
    SELECT target_table,
           ROUND(AVG(execution_time_ms), 1) AS avg_ms,
           MAX(execution_time_ms)           AS max_ms,
           COUNT(*)                         AS n
    FROM   raw_logs
    GROUP  BY target_table
    ORDER  BY avg_ms DESC
''').pl())

print()
print("── Anomaly 2: evening vs daytime ──")
print(conn.execute('''
    SELECT CASE WHEN HOUR(CAST(timestamp AS TIMESTAMP)) BETWEEN 18 AND 22
                THEN 'Evening (18-22h)' ELSE 'Daytime' END AS period,
           ROUND(AVG(execution_time_ms), 1) AS avg_ms,
           COUNT(*)                         AS n
    FROM   raw_logs
    GROUP  BY 1
''').pl())

conn.close()

Total rows  : 15,000
Timeouts    : 339  (2.26%)

── Anomaly 1: orders table vs others ──
shape: (5, 4)
┌───────────────┬────────┬────────┬──────┐
│ target_table  ┆ avg_ms ┆ max_ms ┆ n    │
│ ---           ┆ ---    ┆ ---    ┆ ---  │
│ str           ┆ f64    ┆ i64    ┆ i64  │
╞═══════════════╪════════╪════════╪══════╡
│ orders        ┆ 739.7  ┆ 9975   ┆ 3029 │
│ order_items   ┆ 214.2  ┆ 566    ┆ 2935 │
│ order_reviews ┆ 210.6  ┆ 563    ┆ 3093 │
│ products      ┆ 210.1  ┆ 560    ┆ 3013 │
│ customers     ┆ 208.9  ┆ 560    ┆ 2930 │
└───────────────┴────────┴────────┴──────┘

── Anomaly 2: evening vs daytime ──
shape: (2, 3)
┌──────────────────┬────────┬───────┐
│ period           ┆ avg_ms ┆ n     │
│ ---              ┆ ---    ┆ ---   │
│ str              ┆ f64    ┆ i64   │
╞══════════════════╪════════╪═══════╡
│ Evening (18-22h) ┆ 398.7  ┆ 3134  │
│ Daytime          ┆ 296.3  ┆ 11866 │
└──────────────────┴────────┴───────┘
